# OSINT & Security — Threat Simulation
Models attack surface scoring, CVE risk distribution, and multi-stage attack chain probability.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.stats import beta as beta_dist

plt.style.use('dark_background')
plt.rcParams.update({'axes.facecolor': '#0A0A0C', 'figure.facecolor': '#070708',
                     'text.color': '#F0EDE6', 'axes.labelcolor': '#C8A97E',
                     'xtick.color': '#4A4A56', 'ytick.color': '#4A4A56',
                     'axes.edgecolor': '#252528', 'grid.color': '#1A1A1E'})

rng = np.random.default_rng(99)
print('OSINT threat simulation ready ✓')

In [ ]:
# ── 1. CVE Risk Score Distribution ───────────────────────────────────────────
N_CVE = 500
cve_scores = np.concatenate([
    rng.beta(2, 8, int(N_CVE * 0.45)) * 4,          # Low (0-4)
    rng.beta(5, 5, int(N_CVE * 0.30)) * 3 + 4,      # Medium (4-7)
    rng.beta(8, 3, int(N_CVE * 0.18)) * 2 + 7,      # High (7-9)
    rng.beta(15, 2, int(N_CVE * 0.07)) * 1 + 9,     # Critical (9-10)
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('OSINT Intelligence Suite — Threat Simulation', color='#C8A97E', fontsize=13, fontweight='bold')

ax = axes[0]
severity_colors = ['#28C840', '#FEBC2E', '#FF8C00', '#FF5F57']
bins = [0, 4, 7, 9, 10.01]
labels = ['Low', 'Medium', 'High', 'Critical']
counts = [((cve_scores >= bins[i]) & (cve_scores < bins[i+1])).sum() for i in range(4)]
ax.bar(labels, counts, color=severity_colors, alpha=0.85, edgecolor='none')
ax.set_title(f'CVE Severity ({N_CVE} vulnerabilities)', color='#F0EDE6')
ax.set_ylabel('Count'); ax.grid(True, alpha=0.2, axis='y')
for i, (l, c) in enumerate(zip(labels, counts)):
    ax.text(i, c + 3, f'{c/N_CVE*100:.0f}%', ha='center', color='#8A8A7A', fontsize=9)

print(f'CVE distribution: {dict(zip(labels, counts))}')

In [ ]:
# ── 2. IP Reputation Clustering ──────────────────────────────────────────────
N_IPS = 300
vt_malicious  = rng.integers(0, 80, N_IPS)
otx_pulses    = rng.integers(0, 20, N_IPS)
abuse_score   = np.clip(vt_malicious * 1.5 + otx_pulses * 3 + rng.normal(0, 5, N_IPS), 0, 100)

# Risk score (matches osint.service.ts scoring)
risk = np.clip(vt_malicious * 5 + otx_pulses * 2, 0, 100)

ax = axes[1]
sc = ax.scatter(vt_malicious, otx_pulses, c=risk, cmap='hot', s=18, alpha=0.7, edgecolors='none')
ax.set_title('IP Reputation Space', color='#F0EDE6')
ax.set_xlabel('VirusTotal Detections'); ax.set_ylabel('OTX Threat Pulses')
plt.colorbar(sc, ax=ax, label='Risk Score')
ax.grid(True, alpha=0.15)

# Thresholds
ax.axvline(x=10, color='#FEBC2E', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=5,  color='#FF5F57', linestyle='--', alpha=0.5, linewidth=1)

high_risk = (risk >= 70).sum()
print(f'High-risk IPs (score ≥70): {high_risk} / {N_IPS} ({high_risk/N_IPS*100:.1f}%)')

In [ ]:
# ── 3. Attack Chain Graph (kill-chain simulation) ────────────────────────────
stages = [
    'Reconnaissance', 'Weaponization', 'Delivery',
    'Exploitation', 'Installation', 'C2', 'Exfiltration'
]

# Success probability at each stage (decreases with defenses)
base_p    = [0.95, 0.80, 0.65, 0.42, 0.30, 0.22, 0.15]
defended_p = [0.95, 0.75, 0.40, 0.18, 0.10, 0.06, 0.03]

N_SIM = 10_000
def simulate_chain(probs, n=N_SIM):
    chains = np.ones(n, dtype=bool)
    reached = [n]
    for p in probs:
        chains &= rng.random(n) < p
        reached.append(int(chains.sum()))
    return reached

base_reach    = simulate_chain(base_p)
defense_reach = simulate_chain(defended_p)

ax = axes[2]
x  = range(len(stages) + 1)
ax.fill_between(x, [r/N_SIM*100 for r in base_reach], alpha=0.2, color='#FF5F57')
ax.fill_between(x, [r/N_SIM*100 for r in defense_reach], alpha=0.2, color='#28C840')
ax.plot(x, [r/N_SIM*100 for r in base_reach], 'o-', color='#FF5F57', linewidth=2, label='Undefended')
ax.plot(x, [r/N_SIM*100 for r in defense_reach], 's-', color='#28C840', linewidth=2, label='Axira AV + OSINT')
ax.set_xticks(x)
ax.set_xticklabels(['Start'] + stages, rotation=30, ha='right', fontsize=7)
ax.set_title(f'Kill-Chain Success Rate ({N_SIM:,} simulations)', color='#F0EDE6')
ax.set_ylabel('% Attacks Reaching Stage')
ax.legend(fontsize=8, framealpha=0.3); ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('threat_sim.png', dpi=150, bbox_inches='tight')
plt.show()

reduction = (base_reach[-1] - defense_reach[-1]) / base_reach[-1] * 100
print(f'\nDefense effectiveness: {reduction:.1f}% reduction in successful exfiltration')
print(f'Undefended exfiltration rate: {base_reach[-1]/N_SIM*100:.2f}%')
print(f'Defended  exfiltration rate:  {defense_reach[-1]/N_SIM*100:.2f}%')

In [ ]:
# ── 4. Network Attack Graph ───────────────────────────────────────────────────
G = nx.DiGraph()

nodes = {
    'Internet':    {'risk': 100, 'type': 'external'},
    'Cloudflare':  {'risk': 5,   'type': 'defense'},
    'API Gateway': {'risk': 30,  'type': 'service'},
    'Auth Service':{'risk': 45,  'type': 'service'},
    'News API':    {'risk': 20,  'type': 'service'},
    'PostgreSQL':  {'risk': 35,  'type': 'database'},
    'Redis':       {'risk': 25,  'type': 'database'},
    'Axira AV':    {'risk': 5,   'type': 'defense'},
    'Admin Panel': {'risk': 70,  'type': 'service'},
}
edges = [
    ('Internet', 'Cloudflare', 0.9),
    ('Cloudflare', 'API Gateway', 0.6),
    ('API Gateway', 'Auth Service', 0.4),
    ('API Gateway', 'News API', 0.3),
    ('Auth Service', 'PostgreSQL', 0.25),
    ('News API', 'PostgreSQL', 0.2),
    ('News API', 'Redis', 0.15),
    ('Internet', 'Admin Panel', 0.05),  # direct attack vector
    ('Admin Panel', 'PostgreSQL', 0.6),
    ('Axira AV', 'API Gateway', 0.0),   # monitoring edge
]

for n, d in nodes.items(): G.add_node(n, **d)
for u, v, w in edges: G.add_edge(u, v, weight=w)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#070708'); ax.set_facecolor('#0A0A0C')

pos = nx.spring_layout(G, seed=7, k=2.0)
risk_vals = [G.nodes[n]['risk'] for n in G.nodes]
type_colors = {'external':'#FF5F57','defense':'#28C840','service':'#5AC8FA','database':'#B48EAD'}
node_colors = [type_colors[G.nodes[n]['type']] for n in G.nodes]

edge_weights = [G[u][v]['weight'] * 4 for u, v in G.edges]
nx.draw_networkx_edges(G, pos, edge_color='#2A2A2E', width=edge_weights, alpha=0.7,
                       arrows=True, arrowsize=15, ax=ax,
                       connectionstyle='arc3,rad=0.1')
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=800, alpha=0.9, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, font_color='#F0EDE6', ax=ax)

legend_patches = [mpatches.Patch(color=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_patches, loc='upper left', fontsize=8, framealpha=0.3)
ax.set_title('AxiraNews Attack Surface Graph — Edge weight = breach probability', color='#C8A97E', pad=12)
ax.axis('off')

plt.tight_layout()
plt.savefig('attack_graph.png', dpi=150, bbox_inches='tight')
plt.show()

# Highest-risk path
try:
    path = nx.shortest_path(G, 'Internet', 'PostgreSQL')
    print(f'Shortest attack path to DB: {" → ".join(path)}')
except nx.NetworkXNoPath:
    print('No direct path found (good!)')